# Heart Data Preprocessing



In [ ]:
from pathlib import Path
import pandas as pd
import joblib

# Get project root
project_root = Path.cwd().resolve().parent if Path.cwd().name == 'utils' else Path.cwd().resolve()

# Define columns
CATEGORICAL_COLS = ['cp', 'restecg', 'slope', 'thal']
NUMERICAL_COLS = ['age', 'sex', 'trestbps', 'chol', 'fbs', 'thalach', 'exang', 'oldpeak', 'ca']

# Load raw data
df = pd.read_csv(project_root / "data" / "heart.csv")

df.head()

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import joblib

# Get project root (parent of utils folder)
project_root = Path.cwd().resolve().parent if Path.cwd().name == 'utils' else Path.cwd().resolve()

# Load raw data
df = pd.read_csv(project_root / "data" / "heart.csv")

df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1.0,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0.0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1.0,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1.0,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1.0,106,0,1.9,1,3,2,0


In [17]:
df.isnull().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     1
thalach     0
exang       0
oldpeak     2
slope       0
ca          0
thal        0
target      0
dtype: int64

In [18]:
# Fill numeric missing values with the median to keep the logic simple.
df= df.fillna(df.median())

df.isnull().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

In [ ]:
# Separate target
y = df['target']
df_features = df.drop('target', axis=1)

# One-hot encoding with get_dummies
df_encoded = pd.get_dummies(df_features, columns=CATEGORICAL_COLS, drop_first=True)

# Save column names
column_names = df_encoded.columns.tolist()

# Scale numerical columns (0-1)
scalers = {}
for col in NUMERICAL_COLS:
    if col in df_encoded.columns:
        min_val = df[col].min()
        max_val = df[col].max()
        scalers[col] = {'min': min_val, 'max': max_val}
        df_encoded[col] = (df_encoded[col] - min_val) / (max_val - min_val)

# Add target back
df_processed = df_encoded.copy()
df_processed['target'] = y.values

# Save preprocessor info
preprocessor = {
    'columns': column_names,
    'scalers': scalers,
    'categorical_cols': CATEGORICAL_COLS,
    'numerical_cols': NUMERICAL_COLS
}
joblib.dump(preprocessor, project_root / "utils" / "heart_preprocessor.joblib")

df_processed.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,0.479167,1.0,0.0,0.292453,0.196347,0.0,0.5,0.740458,0.0,0.161290,1.0,0.50,1.000000,0
1,0.500000,1.0,0.0,0.433962,0.175799,1.0,0.0,0.641221,1.0,0.500000,0.0,0.00,1.000000,0
2,0.854167,1.0,0.0,0.481132,0.109589,0.0,0.5,0.412214,1.0,0.419355,0.0,0.00,1.000000,0
3,0.666667,1.0,0.0,0.509434,0.175799,0.0,0.5,0.687023,0.0,0.000000,1.0,0.25,1.000000,0
4,0.687500,0.0,0.0,0.415094,0.383562,1.0,0.5,0.267176,0.0,0.306452,0.5,0.75,0.666667,0


In [ ]:
df_processed.to_csv(project_root / "data" / "cleaned_data.csv", index=False)
print('cleaned_data.csv saved successfully.')
print(f'Feature columns: {len(column_names)}')
print(f'Columns: {column_names}')

cleaned_data.csv saved successfully.
